# Trader Bias Classifier
Multi-class XGBoost pipeline: feature engineering → train/val/test split → hyperparameter tuning → evaluation

In [2]:
import os
import warnings
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import gc

np.random.seed(42)
warnings.filterwarnings('ignore')

LABEL_MAP = {
    'calm':        0,
    'loss_averse': 1,
    'overtrader':  2,
    'revenge':     3,
}
LABEL_NAMES = ['Calm', 'Loss Averse', 'Overtrader', 'Revenge']

FILE_MAP = {
    'calm':        'trading_datasets/calm_trader.csv',
    'loss_averse': 'trading_datasets/loss_averse_trader.csv',
    'overtrader':  'trading_datasets/overtrader.csv',
    'revenge':     'trading_datasets/revenge_trader.csv',
}

# Train/Val/Test split ratios
TRAIN_RATIO = 0.50
VAL_RATIO   = 0.20
TEST_RATIO  = 0.30

print('Config loaded.')

Config loaded.


## 1. Load & Engineer Features

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-row features using rolling windows and lag statistics.
    All features are computed within the trader's own sequence (no lookahead).
    """
    df = df.copy().sort_values('timestamp').reset_index(drop=True)

    # --- Raw derived columns ---
    df['time_diff_s']   = df['timestamp'].diff().dt.total_seconds().fillna(60)
    df['price_move_pct']= (df['exit_price'] - df['entry_price']).abs() / (df['entry_price'] + 1e-9) * 100
    df['is_win']        = (df['profit_loss'] > 0).astype(int)
    df['is_buy']        = (df['side'] == 'BUY').astype(int)

    # --- Lag features (prev trade) ---
    df['prev_pl']       = df['profit_loss'].shift(1).fillna(0)
    df['prev_quantity'] = df['quantity'].shift(1).fillna(df['quantity'].mean())
    df['qty_change']    = df['quantity'] / (df['prev_quantity'] + 1e-9)   # size escalation
    df['prev_is_loss']  = (df['prev_pl'] < 0).astype(int)

    # --- Rolling windows (5, 10, 20 trades) ---
    for w in [5, 10, 20]:
        df[f'roll_win_rate_{w}']   = df['is_win'].rolling(w, min_periods=1).mean()
        df[f'roll_pl_mean_{w}']    = df['profit_loss'].rolling(w, min_periods=1).mean()
        df[f'roll_pl_std_{w}']     = df['profit_loss'].rolling(w, min_periods=1).std().fillna(0)
        df[f'roll_qty_mean_{w}']   = df['quantity'].rolling(w, min_periods=1).mean()
        df[f'roll_qty_std_{w}']    = df['quantity'].rolling(w, min_periods=1).std().fillna(0)
        df[f'roll_time_mean_{w}']  = df['time_diff_s'].rolling(w, min_periods=1).mean()

    # --- Loss streak ---
    streaks = []
    streak = 0
    for pl in df['profit_loss']:
        streak = streak + 1 if pl < 0 else 0
        streaks.append(streak)
    df['loss_streak'] = streaks

    # --- Size after loss signal (rolling: avg qty on loss-following trades) ---
    df['qty_after_loss'] = df['quantity'].where(df['prev_is_loss'] == 1)
    df['qty_after_loss'] = df['qty_after_loss'].rolling(10, min_periods=1).mean().fillna(df['quantity'].mean())

    # --- Hold time asymmetry (rolling avg move % on wins vs losses) ---
    df['win_move']  = df['price_move_pct'].where(df['is_win'] == 1)
    df['loss_move'] = df['price_move_pct'].where(df['is_win'] == 0)
    df['win_move']  = df['win_move'].rolling(10, min_periods=1).mean().fillna(df['price_move_pct'].mean())
    df['loss_move'] = df['loss_move'].rolling(10, min_periods=1).mean().fillna(df['price_move_pct'].mean())
    df['hold_asymmetry'] = df['loss_move'] / (df['win_move'] + 1e-9)

    # --- Loss/win ratio (rolling) ---
    df['roll_avg_win_pl']  = df['profit_loss'].where(df['is_win'] == 1).rolling(10, min_periods=1).mean().fillna(0)
    df['roll_avg_loss_pl'] = df['profit_loss'].where(df['is_win'] == 0).abs().rolling(10, min_periods=1).mean().fillna(0)
    df['loss_win_ratio']   = df['roll_avg_loss_pl'] / (df['roll_avg_win_pl'].abs() + 1e-9)

    # --- Balance drawdown ---
    df['cummax_balance'] = df['balance'].cummax()
    df['drawdown']       = df['balance'] - df['cummax_balance']

    # Drop helper columns not used as features
    drop_cols = ['timestamp', 'asset', 'side', 'trader_type',
                 'win_move', 'loss_move', 'qty_after_loss', 'prev_pl', 'prev_quantity']
    feat_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return feat_df


# Load all CSVs, engineer features, label, combine
all_dfs = []
for name, fpath in FILE_MAP.items():
    df = pd.read_csv(fpath, parse_dates=['timestamp'])
    df = df.dropna(subset=['profit_loss', 'quantity', 'entry_price', 'exit_price', 'balance'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['trader_type'] = name
    feat_df = engineer_features(df)
    feat_df['label'] = LABEL_MAP[name]
    all_dfs.append(feat_df)
    print(f'{name:12s}: {len(feat_df):,} rows, {feat_df.shape[1]} features')
    gc.collect()

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal: {len(combined):,} rows, {combined.shape[1]} columns')

calm        : 9,792 rows, 37 features
loss_averse : 9,783 rows, 37 features
overtrader  : 9,800 rows, 37 features
revenge     : 9,803 rows, 37 features

Total: 39,178 rows, 37 columns


## 2. Train / Val / Test Split
Split **per trader** to preserve class balance (50% train, 20% val, 30% test), then shuffle.

In [4]:
def split_per_class(
    df: pd.DataFrame,
    label_col: str = 'label',
    train_ratio: float = 0.50,
    val_ratio:   float = 0.20,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split each class independently (temporal order preserved within each class),
    then shuffle each split to mix classes.
    """
    trains, vals, tests = [], [], []
    for lbl in df[label_col].unique():
        cls_df = df[df[label_col] == lbl].reset_index(drop=True)
        n = len(cls_df)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        trains.append(cls_df.iloc[:n_train])
        vals.append(cls_df.iloc[n_train:n_train + n_val])
        tests.append(cls_df.iloc[n_train + n_val:])

    train = pd.concat(trains).sample(frac=1, random_state=42).reset_index(drop=True)
    val   = pd.concat(vals).sample(frac=1, random_state=42).reset_index(drop=True)
    test  = pd.concat(tests).sample(frac=1, random_state=42).reset_index(drop=True)
    return train, val, test


train_df, val_df, test_df = split_per_class(combined)

FEATURE_COLS = [c for c in combined.columns if c != 'label']

X_train, y_train = train_df[FEATURE_COLS], train_df['label']
X_val,   y_val   = val_df[FEATURE_COLS],   val_df['label']
X_test,  y_test  = test_df[FEATURE_COLS],  test_df['label']

print(f'Train : {len(X_train):,} rows  |  class dist: {dict(y_train.value_counts().sort_index())}')
print(f'Val   : {len(X_val):,} rows  |  class dist: {dict(y_val.value_counts().sort_index())}')
print(f'Test  : {len(X_test):,} rows  |  class dist: {dict(y_test.value_counts().sort_index())}')

Train : 19,588 rows  |  class dist: {0: np.int64(4896), 1: np.int64(4891), 2: np.int64(4900), 3: np.int64(4901)}
Val   : 7,834 rows  |  class dist: {0: np.int64(1958), 1: np.int64(1956), 2: np.int64(1960), 3: np.int64(1960)}
Test  : 11,756 rows  |  class dist: {0: np.int64(2938), 1: np.int64(2936), 2: np.int64(2940), 3: np.int64(2942)}


## 3. Hyperparameter Tuning on Val Set

In [ ]:
import itertools

param_grid = {
    'max_depth':      [6],
    'learning_rate':  [0.1],
    'subsample':      [1.0],
    'n_estimators':   [300],
}

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f'Sweeping {len(combos)} hyperparameter combinations...')

best_acc    = -np.inf
best_params = None

for combo in combos:
    params = dict(zip(keys, combo))
    model = XGBClassifier(
        **params,
        colsample_bytree=0.8,
        tree_method='hist',
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    acc = (model.predict(X_val) == y_val.values).mean()
    if acc > best_acc:
        best_acc    = acc
        best_params = params

print(f'\nBest val accuracy : {best_acc:.4f}')
print(f'Best params       : {best_params}')

Sweeping 16 hyperparameter combinations...


KeyboardInterrupt: 

## 4. Final Model — Train on Train+Val, Evaluate on Test

In [ ]:
# Combine train + val for final fit
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

final_model = XGBClassifier(
    **best_params,
    colsample_bytree=0.8,
    tree_method='hist',
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_trainval, y_trainval, verbose=False)

y_pred      = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)   # shape (n, 4)

test_acc = (y_pred == y_test.values).mean()
print(f'Test accuracy: {test_acc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

## 6. Confusion Matrix

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333333',
    'axes.labelcolor':  '#cccccc',
    'xtick.color':      '#cccccc',
    'ytick.color':      '#cccccc',
    'text.color':       '#cccccc',
    'font.family':      'monospace',
})

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = final_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(feat_imp) * 0.3)))
colors = ['#e74c3c' if v > feat_imp.median() else '#3498db' for v in feat_imp]
feat_imp.plot.barh(ax=ax, color=colors, edgecolor='#333')
ax.set_title('Feature Importances (final model)', fontsize=13)
ax.set_xlabel('Importance')
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

## 8. Per-Trade Bias Probability Over Time
For each trader type in the test set, plot the predicted probability of each class over trade sequence.

In [ ]:
COLORS = {
    0: '#2ecc71',  # calm
    1: '#e74c3c',  # loss averse
    2: '#f39c12',  # overtrader
    3: '#9b59b6',  # revenge
}

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=False)

for ax, true_lbl in zip(axes, range(4)):
    mask = y_test.values == true_lbl
    probs = y_pred_prob[mask]          # (n_trades_for_this_class, 4)

    for cls in range(4):
        ax.plot(probs[:, cls], color=COLORS[cls], label=LABEL_NAMES[cls],
                linewidth=0.9, alpha=0.85)

    ax.set_title(f'True Class: {LABEL_NAMES[true_lbl]}', fontsize=11)
    ax.set_ylabel('P(class)')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', facecolor='#1a1a1a', edgecolor='#444', fontsize=8)
    ax.grid(True)

axes[-1].set_xlabel('Trade # (within test set for this class)')
fig.suptitle('Per-Trade Bias Probabilities Over Time', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Decision Tree Visualization

In [ ]:
from xgboost import plot_tree

for class_idx, class_name in enumerate(LABEL_NAMES):
    fig, ax = plt.subplots(figsize=(30, 12))
    plot_tree(final_model, num_trees=class_idx, ax=ax, rankdir='LR')
    ax.set_title(f'XGBoost — Tree {class_idx} ({class_name})', fontsize=14, pad=12)
    plt.tight_layout()
    fname = f'decision_tree_{class_name.lower().replace(" ", "_")}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')